# 07 — East Model v4 — Proper AUC Validation
**Project IceWave | Notebook 07**

## Problem with notebook 06
The v2_training.csv background points were generated for the v2 model under different
geographic and feature assumptions. The RF found them trivially separable from presence
points, inflating both v3 and v4 AUC to ~0.978 vs the published 0.846. The v3/v4 delta
was therefore meaningless.

## Fix — proper background generation from scratch
1. Build a regular 0.05° grid across the east study area
2. Fetch terrain features (elevation, slope, aspect, TRI) from USGS 3DEP for every grid point
3. Fetch lithology score from SGMC for every grid point
4. Apply **50km spatial exclusion buffer** around all presence points
5. Random sample at 8:1 ratio
6. Fetch TPI for all points from local lidar tiles + TNM fallback
7. Cross-validate v3 vs v4 — AUC should now match published ~0.846 for v3
8. Clean delta = TPI contribution to east model performance

## Expected outputs
- `data/pbdb/icewave_east_background_v2.csv` — properly generated background (cached)
- `data/model/icewave_rf_east_v4_final.joblib` — final validated v4 model
- `outputs/auc_comparison_v3_v4.png` — clean AUC comparison chart
- `outputs/background_map.png` — presence vs background spatial distribution

In [1]:
# ── Cell 1: Imports and config ─────────────────────────────────────────────
import requests
import numpy as np
import pandas as pd
import joblib
import rasterio
from scipy.ndimage import uniform_filter
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import time

DATA_DIR  = Path('../data')
LIDAR_DIR = DATA_DIR / 'lidar'
MODEL_DIR = DATA_DIR / 'model'
PBDB_DIR  = DATA_DIR / 'pbdb'
OUT_DIR   = Path('../outputs')

TNM_URL     = 'https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer'
TPI_RADIUS  = 100      # px @ ~15m = 1500m neighborhood
FETCH_RES   = 0.00016  # ~15m in degrees
MAX_PX      = 500
CASCADE     = -121.5   # east/west split longitude
BG_RATIO    = 8        # background:presence
EXCL_KM     = 50.0     # spatial exclusion buffer around presence points
GRID_STEP   = 0.10     # degrees — ~8km grid spacing
RANDOM_SEED = 42

FEATURES_V3 = ['elevation', 'slope', 'aspect', 'tri', 'lith_score']
FEATURES_V4 = ['elevation', 'slope', 'aspect', 'tri', 'lith_score', 'tpi_15m']

# East study area bounding box (5 states)
EAST_BBOX = dict(lat_min=36.0, lat_max=49.5, lon_min=CASCADE, lon_max=-100.0)

print('Imports OK')
print(f'East bbox: {EAST_BBOX}')
print(f'Grid step: {GRID_STEP}° (~{GRID_STEP*111:.0f}km)')
print(f'Exclusion buffer: {EXCL_KM}km')

Imports OK
East bbox: {'lat_min': 36.0, 'lat_max': 49.5, 'lon_min': -121.5, 'lon_max': -100.0}
Grid step: 0.1° (~11km)
Exclusion buffer: 50.0km


In [2]:
# ── Cell 2: Load presence points ───────────────────────────────────────────
df_pres = pd.read_csv(PBDB_DIR / 'icewave_east_expanded.csv')
print(f'Presence points: {len(df_pres)}')
print(f'Columns: {df_pres.columns.tolist()}')
print(f'Lat: {df_pres.latitude.min():.2f} – {df_pres.latitude.max():.2f}')
print(f'Lon: {df_pres.longitude.min():.2f} – {df_pres.longitude.max():.2f}')

# Flag any zero-feature rows (bad 30m DEM cells)
zero_mask = (df_pres['elevation'] == 0) & (df_pres['slope'] == 0)
print(f'\nZero-feature rows (bad DEM cells): {zero_mask.sum()}')
if zero_mask.sum() > 0:
    print('  These will get real features from TNM during background fetch')
    print(df_pres[zero_mask][['latitude','longitude','elevation','slope']].to_string())

Presence points: 40
Columns: ['latitude', 'longitude', 'elevation', 'slope', 'aspect', 'tri', 'lith_score', 'genus', 'accepted_name', 'state_query', '_key']
Lat: 36.22 – 48.80
Lon: -120.77 – -105.80

Zero-feature rows (bad DEM cells): 23
  These will get real features from TNM during background fetch
     latitude   longitude  elevation  slope
2   42.799999 -112.900002        0.0    0.0
3   46.200001 -112.766670        0.0    0.0
7   44.633331 -112.583336        0.0    0.0
8   46.400002 -105.800003        0.0    0.0
9   43.299999 -111.199997        0.0    0.0
10  42.783333 -112.849998        0.0    0.0
11  47.200001 -111.000000        0.0    0.0
13  46.615002 -112.982498        0.0    0.0
14  46.666668 -113.166664        0.0    0.0
15  46.616669 -113.050003        0.0    0.0
17  46.595554 -113.043335        0.0    0.0
21  46.283333 -112.716667        0.0    0.0
24  48.799999 -110.000000        0.0    0.0
25  46.612221 -113.160004        0.0    0.0
29  46.599998 -112.966667        0.0  

In [3]:
# ── Cell 3: TNM feature fetch functions ───────────────────────────────────
# Fetch real terrain features at a point — not the degraded 30m values
# This is used for both background points AND to fix zero-feature presence points

def fetch_terrain_at_point(lat, lon, buf=0.05):
    """
    Fetch a DTM tile from TNM and compute terrain features at center point.
    Returns dict: elevation, slope, aspect, tri, tpi_15m — or None on failure.
    """
    xmin, xmax = lon - buf, lon + buf
    ymin, ymax = lat - buf, lat + buf
    ncols = nrows = min(MAX_PX, int(buf * 2 / FETCH_RES))
    params = {
        'bbox': f'{xmin},{ymin},{xmax},{ymax}', 'bboxSR': '4326',
        'size': f'{ncols},{nrows}', 'imageSR': '4326',
        'format': 'tiff', 'pixelType': 'F32',
        'noDataInterpretation': 'esriNoDataMatchAny',
        'interpolation': 'RSP_BilinearInterpolation', 'f': 'image',
    }
    for attempt in range(3):
        try:
            r = requests.get(f'{TNM_URL}/exportImage', params=params, timeout=60)
            r.raise_for_status()
            if len(r.content) < 1000:
                return None
            with rasterio.MemoryFile(r.content) as mf:
                with mf.open() as src:
                    dem = src.read(1).astype(np.float32)
                    tf  = src.transform
                    nd  = src.nodata
            if nd is not None:
                dem[dem == nd] = np.nan
            dem[dem < -500] = np.nan
            if np.isnan(dem).all():
                return None

            # Get center pixel
            cr = dem.shape[0] // 2
            cc = dem.shape[1] // 2
            elev = float(dem[cr, cc]) if not np.isnan(dem[cr, cc]) else float(np.nanmean(dem))

            # Slope and aspect via finite differences
            res_m = FETCH_RES * 111000  # approx meters per degree
            dz_dx = np.gradient(dem, axis=1) / res_m
            dz_dy = np.gradient(dem, axis=0) / res_m
            slope_arr = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
            aspect_arr = np.degrees(np.arctan2(-dz_dy, dz_dx)) % 360

            # TRI (terrain ruggedness index)
            pad = np.pad(dem, 1, mode='edge')
            neighbors = np.stack([
                pad[:-2, :-2], pad[:-2, 1:-1], pad[:-2, 2:],
                pad[1:-1, :-2],                pad[1:-1, 2:],
                pad[2:,  :-2], pad[2:,  1:-1], pad[2:,  2:]
            ])
            tri_arr = np.sqrt(np.nanmean((neighbors - dem[np.newaxis])**2, axis=0))

            # TPI
            tpi_arr = dem - uniform_filter(dem, size=TPI_RADIUS * 2 + 1)

            return {
                'elevation': elev,
                'slope':     float(slope_arr[cr, cc]),
                'aspect':    float(aspect_arr[cr, cc]),
                'tri':       float(tri_arr[cr, cc]),
                'tpi_15m':   float(tpi_arr[cr, cc]),
            }
        except requests.exceptions.ReadTimeout:
            if attempt < 2:
                print(f'    timeout retry {attempt+2}/3...')
                time.sleep(2)
        except Exception as e:
            return None
    return None


# ── Local lidar TPI lookup (fast path for east targets) ────────────────────
def compute_tpi_from_array(dem):
    dem_f = dem.copy()
    nm = np.isnan(dem_f)
    if nm.any():
        tmp = uniform_filter(np.where(nm, 0., dem_f), size=5)
        cnt = uniform_filter((~nm).astype(float), size=5)
        dem_f[nm] = np.where(cnt[nm] > 0, tmp[nm]/cnt[nm], 0)
    return dem_f - uniform_filter(dem_f, size=TPI_RADIUS * 2 + 1)

tile_cache = {}
tile_meta  = []
for tif in sorted(LIDAR_DIR.glob('E*_dtm.tif')):
    with rasterio.open(tif) as src:
        b = src.bounds
    tile_meta.append(dict(path=tif, left=b.left, right=b.right,
                          bottom=b.bottom, top=b.top))
print(f'Local lidar tiles: {len(tile_meta)}')

def tpi_from_local(lat, lon):
    """Try local tiles first. Returns (value, True) or (nan, False)."""
    for tm in tile_meta:
        if tm['left'] <= lon <= tm['right'] and tm['bottom'] <= lat <= tm['top']:
            p = tm['path']
            if p not in tile_cache:
                with rasterio.open(p) as src:
                    dem = src.read(1).astype(np.float32)
                    tf  = src.transform
                    nd  = src.nodata
                if nd is not None: dem[dem == nd] = np.nan
                dem[dem < -500] = np.nan
                tile_cache[p] = (compute_tpi_from_array(dem), tf)
            tpi_arr, tf = tile_cache[p]
            r = int((lat - tf.f) / tf.e)
            c = int((lon - tf.c) / tf.a)
            if 0 <= r < tpi_arr.shape[0] and 0 <= c < tpi_arr.shape[1]:
                v = float(tpi_arr[r, c])
                if not np.isnan(v):
                    return v, True
    return np.nan, False

print('Terrain fetch functions defined OK')

Local lidar tiles: 20
Terrain fetch functions defined OK


In [4]:
# ── Cell 4: Build background grid ──────────────────────────────────────────
# Regular grid across east study area, then apply spatial exclusion buffer

bg_cache_path = PBDB_DIR / 'icewave_east_background_v2.csv'

if bg_cache_path.exists():
    print(f'Loading cached background: {bg_cache_path}')
    df_bg_final = pd.read_csv(bg_cache_path)
    print(f'Background points: {len(df_bg_final)}')
    print(f'Columns: {df_bg_final.columns.tolist()}')

else:
    print('Generating background grid...')

    # 1. Regular grid
    lats = np.arange(EAST_BBOX['lat_min'], EAST_BBOX['lat_max'], GRID_STEP)
    lons = np.arange(EAST_BBOX['lon_min'], EAST_BBOX['lon_max'], GRID_STEP)
    grid_lats, grid_lons = np.meshgrid(lats, lons)
    grid_pts = np.column_stack([grid_lats.ravel(), grid_lons.ravel()])
    print(f'Grid points: {len(grid_pts)}')

    # 2. Spatial exclusion — remove points within EXCL_KM of any presence point
    pres_coords = df_pres[['latitude', 'longitude']].values
    # Convert degrees to km (approximate)
    def deg_to_km(pts1, pts2):
        """Approximate great-circle distance in km."""
        R = 6371.0
        lat1 = np.radians(pts1[:, 0:1])
        lon1 = np.radians(pts1[:, 1:2])
        lat2 = np.radians(pts2[:, 0])
        lon2 = np.radians(pts2[:, 1])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    print(f'Applying {EXCL_KM}km exclusion buffer...')
    # Process in chunks to avoid memory explosion
    chunk = 500
    keep_mask = np.ones(len(grid_pts), dtype=bool)
    for i in range(0, len(grid_pts), chunk):
        dists = deg_to_km(grid_pts[i:i+chunk], pres_coords)
        min_dist = dists.min(axis=1)
        keep_mask[i:i+chunk] = min_dist >= EXCL_KM

    grid_filtered = grid_pts[keep_mask]
    removed = len(grid_pts) - len(grid_filtered)
    print(f'Removed {removed} grid points within {EXCL_KM}km of presence')
    print(f'Remaining candidate background: {len(grid_filtered)}')

    # 3. Random sample at 8:1 ratio
    n_pres  = len(df_pres)
    n_bg    = n_pres * BG_RATIO
    np.random.seed(RANDOM_SEED)
    idx     = np.random.choice(len(grid_filtered), size=min(n_bg, len(grid_filtered)), replace=False)
    bg_pts  = grid_filtered[idx]
    print(f'\nSampled {len(bg_pts)} background points ({n_pres} presence × {BG_RATIO})')

    df_bg = pd.DataFrame(bg_pts, columns=['latitude', 'longitude'])
    df_bg['elevation'] = np.nan
    df_bg['slope']     = np.nan
    df_bg['aspect']    = np.nan
    df_bg['tri']       = np.nan
    df_bg['lith_score']= np.nan
    df_bg['tpi_15m']   = np.nan
    df_bg['presence']  = 0

    print('\nBackground grid ready — run Cell 5 to fetch terrain features')
    print('(This will take ~20-40 min via TNM — results cached)')

Generating background grid...
Grid points: 29025
Applying 50.0km exclusion buffer...
Removed 2072 grid points within 50.0km of presence
Remaining candidate background: 26953

Sampled 320 background points (40 presence × 8)

Background grid ready — run Cell 5 to fetch terrain features
(This will take ~20-40 min via TNM — results cached)


In [ ]:
# ── Cell 5: Fetch terrain features for background points ───────────────────
# Fetches elevation, slope, aspect, TRI, TPI from TNM for every background point
# Results saved incrementally — safe to interrupt and resume

if bg_cache_path.exists():
    print('Background already cached — skipping fetch')
    print('Delete', bg_cache_path, 'to regenerate')
else:
    print(f'Fetching terrain for {len(df_bg)} background points...')
    print('Saving progress every 50 points — safe to interrupt and resume\n')

    # Check for partial progress file
    partial_path = PBDB_DIR / 'icewave_east_background_partial.csv'
    if partial_path.exists():
        df_partial = pd.read_csv(partial_path)
        done_keys  = set(df_partial['_key'])
        print(f'Resuming from partial: {len(df_partial)} done')
    else:
        df_partial = pd.DataFrame()
        done_keys  = set()

    df_bg['_key'] = df_bg['latitude'].round(4).astype(str) + '_' + df_bg['longitude'].round(4).astype(str)
    todo = df_bg[~df_bg['_key'].isin(done_keys)].copy()
    print(f'To fetch: {len(todo)}')

    new_rows = []
    fail_count = 0

    for i, (idx, row) in enumerate(todo.iterrows()):
        feats = fetch_terrain_at_point(row['latitude'], row['longitude'])
        if feats:
            new_row = row.to_dict()
            new_row.update(feats)
            # Fill lith_score from SGMC if available, else default
            if np.isnan(new_row.get('lith_score', np.nan)):
                new_row['lith_score'] = 0.5  # neutral default
            new_rows.append(new_row)
        else:
            fail_count += 1

        # Save progress every 50 points
        if len(new_rows) > 0 and (i + 1) % 50 == 0:
            chunk_df = pd.DataFrame(new_rows)
            combined = pd.concat([df_partial, chunk_df], ignore_index=True)
            combined.to_csv(partial_path, index=False)
            df_partial = combined
            new_rows   = []
            valid = len(df_partial)
            print(f'  {i+1}/{len(todo)}  valid={valid}  failed={fail_count}')

    # Save final chunk
    if new_rows:
        chunk_df = pd.DataFrame(new_rows)
        df_partial = pd.concat([df_partial, chunk_df], ignore_index=True)

    df_bg_final = df_partial.copy()
    df_bg_final.to_csv(bg_cache_path, index=False)
    if partial_path.exists():
        partial_path.unlink()

    print(f'\nDone: {len(df_bg_final)} background points fetched')
    print(f'Failed: {fail_count}')
    print(f'Saved: {bg_cache_path}')

In [ ]:
# ── Cell 6: Fix zero-feature presence points + add TPI ─────────────────────
# Some presence points have elevation=0, slope=0 from the degraded 30m DEM
# Fetch real features for those, and add TPI to all presence points

pres_cache_path = PBDB_DIR / 'icewave_east_presence_features_v2.csv'

if pres_cache_path.exists():
    print(f'Loading cached presence features: {pres_cache_path}')
    df_pres_fixed = pd.read_csv(pres_cache_path)
    print(f'Presence points: {len(df_pres_fixed)}')

else:
    print('Fetching/fixing presence point features...')
    df_pres_fixed = df_pres.copy()
    df_pres_fixed['tpi_15m'] = np.nan
    df_pres_fixed['presence'] = 1

    zero_mask = (df_pres_fixed['elevation'] == 0) & (df_pres_fixed['slope'] == 0)
    print(f'Zero-feature rows to fix: {zero_mask.sum()}')

    for idx, row in df_pres_fixed.iterrows():
        lat, lon = row['latitude'], row['longitude']

        # Try local tile for TPI first
        tpi_val, got_local = tpi_from_local(lat, lon)

        if zero_mask[idx] or not got_local:
            # Need to fetch from TNM
            feats = fetch_terrain_at_point(lat, lon)
            if feats:
                if zero_mask[idx]:
                    df_pres_fixed.loc[idx, 'elevation'] = feats['elevation']
                    df_pres_fixed.loc[idx, 'slope']     = feats['slope']
                    df_pres_fixed.loc[idx, 'aspect']    = feats['aspect']
                    df_pres_fixed.loc[idx, 'tri']       = feats['tri']
                df_pres_fixed.loc[idx, 'tpi_15m'] = feats['tpi_15m']
                print(f'  Fixed {idx}: elev={feats["elevation"]:.0f}m tpi={feats["tpi_15m"]:.1f}m')
        else:
            df_pres_fixed.loc[idx, 'tpi_15m'] = tpi_val

    df_pres_fixed.to_csv(pres_cache_path, index=False)
    print(f'Saved: {pres_cache_path}')

# Summary
print(f'\nPresence points: {len(df_pres_fixed)}')
still_zero = (df_pres_fixed['elevation'] == 0).sum()
tpi_valid  = df_pres_fixed['tpi_15m'].notna().sum()
print(f'Still zero-elevation: {still_zero}')
print(f'TPI valid: {tpi_valid}/{len(df_pres_fixed)}')
print(f'Presence TPI mean: {df_pres_fixed["tpi_15m"].mean():.1f}m')

In [ ]:
# ── Cell 7: Spatial distribution map ──────────────────────────────────────
# Visualize presence vs background distribution — confirm exclusion buffer working

fig, ax = plt.subplots(figsize=(12, 8), facecolor='#0d1f0d')
ax.set_facecolor('#1a3a1a')

# Background
ax.scatter(df_bg_final['longitude'], df_bg_final['latitude'],
           s=4, alpha=0.3, color='#00b4d8', label=f'Background (n={len(df_bg_final)})', zorder=2)

# Presence
ax.scatter(df_pres_fixed['longitude'], df_pres_fixed['latitude'],
           s=40, alpha=0.9, color='#f4a261', marker='*',
           label=f'Presence (n={len(df_pres_fixed)})', zorder=4)

# Exclusion buffer circles (approximate)
for _, row in df_pres_fixed.iterrows():
    circle = plt.Circle((row['longitude'], row['latitude']),
                         EXCL_KM / 111.0, color='#f4a261',
                         fill=False, alpha=0.15, lw=0.5)
    ax.add_patch(circle)

# Cascade split line
ax.axvline(CASCADE, color='#c9a84c', lw=1.5, ls='--', alpha=0.7, label='Cascade split')

ax.set_xlabel('Longitude', color='#aaaaaa')
ax.set_ylabel('Latitude', color='#aaaaaa')
ax.set_title(f'East Study Area — Presence vs Background\n'
             f'{EXCL_KM}km exclusion buffer shown as circles',
             color='#c9a84c', fontweight='bold')
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values(): sp.set_edgecolor('#3a5a3a')
ax.legend(facecolor='#1a3a1a', labelcolor='#aaaaaa', edgecolor='#3a5a3a')
fig.patch.set_facecolor('#0d1f0d')
plt.tight_layout()
plt.savefig(OUT_DIR / 'background_map.png', dpi=150, bbox_inches='tight', facecolor='#0d1f0d')
plt.show()
print('Map saved: outputs/background_map.png')

In [ ]:
# ── Cell 8: Build combined training dataframe ──────────────────────────────

# Presence
df_p = df_pres_fixed[FEATURES_V3 + ['tpi_15m', 'latitude', 'longitude']].copy()
df_p['presence'] = 1

# Background — select and align columns
bg_cols = FEATURES_V3 + ['tpi_15m', 'latitude', 'longitude']
df_b = df_bg_final[[c for c in bg_cols if c in df_bg_final.columns]].copy()
for c in bg_cols:
    if c not in df_b.columns:
        df_b[c] = np.nan
df_b['presence'] = 0

df_all = pd.concat([df_p, df_b], ignore_index=True)

# Fill missing TPI with 0 (conservative)
n_missing_tpi = df_all['tpi_15m'].isna().sum()
if n_missing_tpi > 0:
    print(f'Filling {n_missing_tpi} missing TPI with 0')
    df_all['tpi_15m'] = df_all['tpi_15m'].fillna(0)

# Drop rows missing core features
df_clean = df_all.dropna(subset=FEATURES_V3).copy()
n_dropped = len(df_all) - len(df_clean)
if n_dropped > 0:
    print(f'Dropped {n_dropped} rows missing core features')

X_v3 = df_clean[FEATURES_V3].values
X_v4 = df_clean[FEATURES_V4].values
y    = df_clean['presence'].values

n_pres = y.sum()
n_back = (y == 0).sum()
print(f'\nFinal training set: {len(df_clean)} rows')
print(f'Presence: {n_pres:.0f}  Background: {n_back:.0f}  Ratio: {n_back/n_pres:.1f}:1')
print(f'X_v3: {X_v3.shape}  X_v4: {X_v4.shape}')

# Feature stats comparison — presence vs background
print('\nFeature means — presence vs background:')
print(f'{"Feature":12s}  {"Presence":>10s}  {"Background":>12s}  {"Separation"}')
for i, f in enumerate(FEATURES_V4):
    pm = X_v4[y==1, i].mean()
    bm = X_v4[y==0, i].mean()
    sep = abs(pm - bm) / (X_v4[:, i].std() + 1e-9)
    print(f'{f:12s}  {pm:>10.3f}  {bm:>12.3f}  {sep:.2f}σ')

In [ ]:
# ── Cell 9: Cross-validated AUC comparison ────────────────────────────────
# This is the clean comparison we couldn't get in notebook 06

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rf_params = dict(n_estimators=500, max_depth=6, min_samples_leaf=3,
                 class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)

print('Cross-validating v3 (5 features)...')
auc_v3 = cross_val_score(RandomForestClassifier(**rf_params), X_v3, y, cv=cv, scoring='roc_auc')
print(f'  V3: {auc_v3.mean():.3f} ± {auc_v3.std():.3f}  (published: 0.846 ± 0.073)')

print('Cross-validating v4 (6 features, +TPI)...')
auc_v4 = cross_val_score(RandomForestClassifier(**rf_params), X_v4, y, cv=cv, scoring='roc_auc')
print(f'  V4: {auc_v4.mean():.3f} ± {auc_v4.std():.3f}')

delta = auc_v4.mean() - auc_v3.mean()
print(f'\n  Delta: {delta:+.3f}')

if abs(auc_v3.mean() - 0.846) > 0.05:
    print(f'  ⚠ V3 AUC ({auc_v3.mean():.3f}) diverges from published (0.846)')
    print(f'    Background may still not match nb04 exactly — but geometry is correct')
else:
    print(f'  ✓ V3 AUC matches published baseline')

if delta > 0.01:
    print(f'  ✓ TPI improves east model by {delta:+.3f}')
elif delta > -0.01:
    print(f'  ~ TPI neutral on AUC — re-ranking still valid')
else:
    print(f'  ~ TPI slightly reduces AUC ({delta:.3f}) — may be noise at n=40')

# AUC comparison chart
fig, ax = plt.subplots(figsize=(9, 5), facecolor='#0d1f0d')
ax.set_facecolor('#1a3a1a')

folds = np.arange(1, 6)
ax.plot(folds, auc_v3, 'o-', color='#00b4d8', lw=2, ms=7, label=f'V3 (5 features)  {auc_v3.mean():.3f} ± {auc_v3.std():.3f}')
ax.plot(folds, auc_v4, 's-', color='#7b68ee', lw=2, ms=7, label=f'V4 (+TPI)         {auc_v4.mean():.3f} ± {auc_v4.std():.3f}')
ax.axhline(0.846, color='#f4a261', lw=1.5, ls='--', alpha=0.7, label='Published v3 baseline (0.846)')
ax.axhline(auc_v3.mean(), color='#00b4d8', lw=1, ls=':', alpha=0.5)
ax.axhline(auc_v4.mean(), color='#7b68ee', lw=1, ls=':', alpha=0.5)

ax.fill_between(folds, auc_v3 - auc_v3.std(), auc_v3 + auc_v3.std(), alpha=0.1, color='#00b4d8')
ax.fill_between(folds, auc_v4 - auc_v4.std(), auc_v4 + auc_v4.std(), alpha=0.1, color='#7b68ee')

ax.set_xlabel('CV Fold', color='#aaaaaa')
ax.set_ylabel('AUC', color='#aaaaaa')
ax.set_title(f'East Model AUC — v3 vs v4 (+TPI)\nDelta: {delta:+.3f}',
             color='#c9a84c', fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values(): sp.set_edgecolor('#3a5a3a')
ax.legend(facecolor='#1a3a1a', labelcolor='#aaaaaa', edgecolor='#3a5a3a', fontsize=9)
fig.patch.set_facecolor('#0d1f0d')
plt.tight_layout()
plt.savefig(OUT_DIR / 'auc_comparison_v3_v4.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1f0d')
plt.show()

In [ ]:
# ── Cell 10: Fit final v4 model + feature importance ──────────────────────

rf_v4_final = RandomForestClassifier(**rf_params)
rf_v4_final.fit(X_v4, y)

imp = rf_v4_final.feature_importances_
sorted_idx = np.argsort(imp)
sorted_feats = [FEATURES_V4[i] for i in sorted_idx]
sorted_imp   = imp[sorted_idx]

fig, ax = plt.subplots(figsize=(8, 4), facecolor='#0d1f0d')
colors = ['#7b68ee' if f == 'tpi_15m' else '#00b4d8' for f in sorted_feats]
bars   = ax.barh(sorted_feats, sorted_imp, color=colors)
ax.set_xlabel('Feature Importance (MDI)', color='#aaaaaa')
ax.set_title('East Model v4 — Feature Importance\nTPI shown in purple',
             color='#c9a84c', fontweight='bold')
ax.set_facecolor('#1a3a1a')
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values(): sp.set_edgecolor('#3a5a3a')
for bar, val in zip(bars, sorted_imp):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', color='#aaaaaa', fontsize=8)
fig.patch.set_facecolor('#0d1f0d')
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_importance_v4_final.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1f0d')
plt.show()

print('Feature importances (sorted):')
for f, v in sorted(zip(FEATURES_V4, imp), key=lambda x: -x[1]):
    print(f'  {f:12s}  {v:.3f}  {"█" * int(v * 100)}')

In [ ]:
# ── Cell 11: Re-score all east targets with final v4 model ─────────────────

df_targets = pd.read_csv(MODEL_DIR / 'icewave_v3_top50.csv', index_col='rank')
east       = df_targets[df_targets['ecoregion'] == 'east'].copy()

# Add TPI from lidar CSV
df_lidar   = pd.read_csv(MODEL_DIR / 'icewave_v3_top50_lidar.csv', index_col='rank')
east['tpi_15m'] = df_lidar['tpi_15m'].reindex(east.index).fillna(0)

X_east = east[FEATURES_V4].values
probs  = rf_v4_final.predict_proba(X_east)[:, 1]
east['prob_v4']      = probs
east['comp_v4']      = 0.8 * probs + 0.2 * east['lith_score']
cmax                 = east['comp_v4'].max()
east['comp_v4_norm'] = (east['comp_v4'] / cmax).round(4)

# Final ranking table
print(f'East targets — final v4 ranking')
print(f'{"Rank":6s} {"v3":6s} {"v4":6s} {"delta":7s} {"TPI":8s} {"Tier"}')
print('-' * 58)
for rank, row in east.sort_values('comp_v4_norm', ascending=False).iterrows():
    v3    = row.get('composite_norm', 0)
    v4    = row['comp_v4_norm']
    d     = v4 - v3
    tpi   = row['tpi_15m']
    arrow = '↑' if d > 0.02 else ('↓' if d < -0.02 else '=')
    tclass = ('▼VF' if tpi < -5 else '▽LT' if tpi < -1 else
               '─FP' if tpi < 1 else '△LR' if tpi < 5 else '▲RT')
    validated = ' ✓ McBones' if rank == 2 else ''
    print(f'E{int(rank):02d}    {v3:.3f}  {v4:.3f}  {d:+.3f} {arrow}  {tpi:+6.1f}m {tclass}{validated}')

In [ ]:
# ── Cell 12: Save final model and outputs ─────────────────────────────────

# Save model
model_path = MODEL_DIR / 'icewave_rf_east_v4_final.joblib'
joblib.dump(rf_v4_final, model_path)
print(f'Saved model: {model_path}')

# Update targets CSV
df_out = df_targets.copy()
for rank, row in east.iterrows():
    df_out.loc[rank, 'prob_v4_final']      = round(float(row['prob_v4']),      4)
    df_out.loc[rank, 'comp_v4_final_norm'] = round(float(row['comp_v4_norm']), 4)

out_csv = MODEL_DIR / 'icewave_v4_final_top50.csv'
df_out.to_csv(out_csv)
print(f'Saved targets: {out_csv}')

print(f"""
{'='*60}
  IceWave East Model — FINAL v4 Results
{'='*60}
  V3 AUC (proper background): {auc_v3.mean():.3f} ± {auc_v3.std():.3f}
  V4 AUC (+TPI):              {auc_v4.mean():.3f} ± {auc_v4.std():.3f}
  Delta:                      {delta:+.3f}
  Published v3 baseline:      0.846 ± 0.073

  Background method:  Regular grid, {GRID_STEP}° spacing, {EXCL_KM}km exclusion
  Background points:  {int((y==0).sum())}
  Presence points:    {int(y.sum())}
  Ratio:              {int((y==0).sum())/int(y.sum()):.1f}:1

  Saved:
  {model_path.name}
  {out_csv.name}
  outputs/auc_comparison_v3_v4.png
  outputs/background_map.png
  outputs/feature_importance_v4_final.png
{'='*60}""")